In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 26. LoRAテキスト テキスト — テキスト テキスト テキスト [CPU/GPU ベンチマーク ⑫]

> **学習目標**
> - LoRAテキスト テキスト テキスト テキスト テキスト度テキスト
> - テキスト(Pruning)テキスト テキスト(sparsity)テキスト テキスト テキスト
> - LoRA 学習テキスト テキスト テキスト テキスト テキスト テキスト テキスト

## 26.1 PEFTテキスト テキスト

テキスト テキスト: テキスト テキスト テキスト → テキスト GPU テキスト, テキスト.

**PEFT** (Parameter-Efficient Fine-Tuning): テキスト テキスト.
- LoRA, Adapter, Prefix Tuning, Prompt Tuning
- 1% テキスト テキスト テキスト FTテキスト テキスト 性能

## 26.2 LoRA — テキスト テキスト

テキスト テキスト $\Delta W$テキスト テキスト(low-rank)テキスト テキスト:
$$W' = W + \Delta W = W + B A$$

- $A \in \mathbb{R}^{r \times d}$, $B \in \mathbb{R}^{d \times r}$
- $r \ll d$ (テキスト 4~64)
- テキスト: $d^2 \to 2rd$ (テキスト テキスト)

テキスト:
- $A \sim \mathcal{N}(0, \sigma^2)$ (テキスト)
- $B = 0$ (テキスト $\Delta W = 0$)

テキスト: $\Delta W = \frac{\alpha}{r} B A$

学習: $W$テキスト テキスト, $A, B$テキスト 学習.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

class LoRALinear(nn.Module):
    """LoRAテキスト Applicationテキスト LinearLayer."""
    def __init__(self, in_features, out_features, r=8, alpha=16, bias=True):
        super().__init__()
        self.base = nn.Linear(in_features, out_features, bias=bias)
        # base Weight テキスト
        self.base.weight.requires_grad = False
        if bias:
            self.base.bias.requires_grad = False
        # LoRA テキスト
        self.A = nn.Parameter(torch.randn(r, in_features) * 0.01)
        self.B = nn.Parameter(torch.zeros(out_features, r))
        self.scaling = alpha / r

    def forward(self, x):
        # base + LoRA delta
        base_out = self.base(x)
        lora_out = (x @ self.A.t()) @ self.B.t() * self.scaling
        return base_out + lora_out

# Parameter Count Comparison
d = 4096
full = nn.Linear(d, d)
lora = LoRALinear(d, d, r=8, alpha=16)
full_params = sum(p.numel() for p in full.parameters())
lora_params = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"Full Linear: {full_params:,} params")
print(f"LoRA (r=8): {lora_params:,} trainable params")
print(f"Ratio: {lora_params/full_params*100:.3f}%")
print("\n=> LoRAテキスト 0.4% テキスト Training!")


## 26.3 テキスト $r$テキスト テキスト

$r$テキスト テキスト テキスト ↑, テキスト ↑. テキスト テキスト度テキスト テキスト テキスト $r$ テキスト.
- テキスト テキスト: $r = 4$
- テキスト テキスト: $r = 16, 32, 64$


In [ ]:
# テキスト テキスト テキスト 性能 (テキスト)
ranks = [1, 2, 4, 8, 16, 32, 64, 128]
d = 4096
print(f"{'r':>5} | {'LoRA params':>12} | {'% of full':>10}")
print("-" * 35)
for r in ranks:
    lora_params = 2 * r * d  # A + B
    full_params = d * d
    print(f"{r:>5} | {lora_params:>12,} | {lora_params/full_params*100:>9.3f}%")

# Visualization
fig, ax = plt.subplots(figsize=(9, 5))
params_pct = [2 * r * d / (d * d) * 100 for r in ranks]
ax.plot(ranks, params_pct, 'o-', linewidth=2, markersize=8)
ax.set_xlabel('LoRA rank r')
ax.set_ylabel('Trainable params (%)')
ax.set_title(f'LoRA: rank vs trainable params (d={d})')
ax.set_xscale('log')
ax.set_yscale('log')
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('../figures/ch26_lora_ranks.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.4 テキスト LoRA テキスト

テキスト LoRAテキスト テキスト テキスト:
- $W_q, W_k, W_v, W_o$ (Attention)
- FFNテキスト $W_1, W_2$

テキスト Attentionテキスト Q, Vテキスト テキスト (QLoRA テキスト テキスト). テキスト テキスト 性能テキスト テキスト.


In [ ]:
# Attentionテキスト LoRA テキスト
class LoRAAttention(nn.Module):
    def __init__(self, d_model, n_heads, r=8):
        super().__init__()
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        # base テキスト (テキスト)
        self.W_qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.W_qkv.weight.requires_grad = False
        self.W_o = nn.Linear(d_model, d_model, bias=False)
        self.W_o.weight.requires_grad = False
        # LoRA テキスト (QKVテキスト)
        self.lora_qkv_A = nn.Parameter(torch.randn(r, d_model) * 0.01)
        self.lora_qkv_B = nn.Parameter(torch.zeros(3 * d_model, r))
        self.scaling = 16 / r

    def forward(self, x, mask=None):
        B, T, D = x.shape
        base = self.W_qkv(x)
        # LoRA delta
        delta = (x @ self.lora_qkv_A.t()) @ self.lora_qkv_B.t() * self.scaling
        qkv = base + delta
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        scores = q @ k.transpose(-1, -2) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores + mask
        weights = F.softmax(scores, dim=-1)
        out = (weights @ v).transpose(1, 2).contiguous().view(B, T, D)
        return self.W_o(out), weights

# テキスト
d, h = 64, 4
attn = LoRAAttention(d, h, r=4)
x = torch.randn(2, 10, d)
out, w = attn(x)
print(f"Output: {out.shape}")
n_trainable = sum(p.numel() for p in attn.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in attn.parameters())
print(f"Trainable: {n_trainable:,} / Total: {n_total:,} ({n_trainable/n_total*100:.2f}%)")


## 26.5 テキスト (Pruning)

### テキスト テキスト
テキスト テキスト 0テキスト:
$$\tilde{W}_{ij} = W_{ij} \cdot \mathbb{1}[|W_{ij}| > \tau]$$

問題: テキスト テキスト テキスト テキスト.

### テキスト テキスト
テキスト/テキスト テキスト テキスト:
- テキスト テキスト度テキスト テキスト テキスト (attention head pruning)
- テキスト/テキスト テキスト (CNN)
- テキスト テキスト

**Lottery Ticket Hypothesis**: テキスト テキスト テキスト 性能 テキスト テキスト テキスト テキスト.


In [ ]:
# テキスト テキスト
def magnitude_prune(W, sparsity=0.5):
    """Magnitude テキスト テキストStructureテキスト Pruning."""
    threshold = torch.quantile(W.abs().flatten(), sparsity)
    mask = (W.abs() > threshold).float()
    return W * mask, mask

# Test
torch.manual_seed(0)
W = torch.randn(100, 100) * 0.1
for sparsity in [0.5, 0.7, 0.9, 0.95]:
    W_pruned, mask = magnitude_prune(W, sparsity)
    actual_sparsity = 1 - mask.mean()
    print(f"テキスト sparsity {sparsity}: テキスト {actual_sparsity:.4f}, "
          f"テキスト Circleテキスト {int(mask.sum())}テキスト")

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
W_orig = torch.randn(50, 50) * 0.1
for ax, sp in zip(axes, [0.0, 0.7, 0.9]):
    W_p, _ = magnitude_prune(W_orig, sp)
    ax.imshow(W_p.numpy(), cmap='RdBu', vmin=-0.3, vmax=0.3)
    ax.set_title(f'Sparsity = {sp}')
plt.tight_layout()
plt.savefig('../figures/ch26_pruning.png', dpi=100, bbox_inches='tight')
plt.show()


## 26.6 [CPU/GPU ベンチマーク ⑫] Full FT vs LoRA vs QLoRA


In [ ]:
# Full FT vs LoRA 比較
from llm_math.bench import time_fn

# テキスト モデル (テキスト テキスト)
class TinyModel(nn.Module):
    def __init__(self, d=512):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
    def forward(self, x):
        return self.fc3(F.relu(self.fc2(F.relu(self.fc1(x)))))

# Full FT
def make_full_ft():
    model = TinyModel()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

# LoRA
class TinyModelLoRA(nn.Module):
    def __init__(self, d=512, r=8):
        super().__init__()
        self.fc1 = nn.Linear(d, d * 4)
        self.fc2 = nn.Linear(d * 4, d)
        self.fc3 = nn.Linear(d, d)
        # base テキスト
        for p in self.parameters():
            p.requires_grad = False
        # LoRA テキスト (base fc1/fc2/fc3テキスト LoRA テキスト テキスト テキスト
        # base テキスト deltaテキスト テキスト テキスト)
        self.lora_A1 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B1 = nn.Parameter(torch.zeros(d * 4, r))
        self.lora_A2 = nn.Parameter(torch.randn(r, d * 4) * 0.01)
        self.lora_B2 = nn.Parameter(torch.zeros(d, r))
        self.lora_A3 = nn.Parameter(torch.randn(r, d) * 0.01)
        self.lora_B3 = nn.Parameter(torch.zeros(d, r))
        self.scaling = 16 / r
    def forward(self, x):
        # fc1 + LoRA delta
        h1 = F.relu(self.fc1(x) + (x @ self.lora_A1.t()) @ self.lora_B1.t() * self.scaling)
        # fc2 + LoRA delta
        h2 = F.relu(self.fc2(h1) + (h1 @ self.lora_A2.t()) @ self.lora_B2.t() * self.scaling)
        # fc3 + LoRA delta
        out = self.fc3(h2) + (h2 @ self.lora_A3.t()) @ self.lora_B3.t() * self.scaling
        return out

def make_lora():
    model = TinyModelLoRA(r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

# 学習 テキスト テキスト 時間 比較
def bench_step(model, opt, x, y, loss_fn, device='cpu'):
    def step():
        opt.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        opt.step()
        return loss
    return step

d = 256
loss_fn = nn.MSELoss()
x = torch.randn(8, d)
y = torch.randn(8, d)

# Modelテキスト d=256テキスト Generation
def make_full_ft(d=256):
    model = TinyModel(d=d)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    return model, opt

def make_lora(d=256):
    model = TinyModelLoRA(d=d, r=8)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=1e-4)
    return model, opt

model_full, opt_full = make_full_ft(d=d)
model_lora, opt_lora = make_lora(d=d)

n_full = sum(p.numel() for p in model_full.parameters() if p.requires_grad)
n_lora = sum(p.numel() for p in model_lora.parameters() if p.requires_grad)
print(f"Full FT: {n_full:,} trainable params")
print(f"LoRA:    {n_lora:,} trainable params ({n_lora/n_full*100:.2f}%)")

res_full = time_fn(bench_step(model_full, opt_full, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
res_lora = time_fn(bench_step(model_lora, opt_lora, x, y, loss_fn), device='cpu', warmup=2, repeat=5)
print(f"\nFull FT 1 step: {res_full['mean_ms']:.3f} ms")
print(f"LoRA 1 step:    {res_lora['mean_ms']:.3f} ms")
print(f"Speed Improvement: {res_full['mean_ms'] / res_lora['mean_ms']:.2f}x")
print("\n=> LoRAテキスト Training テキスト テキスト Memoryテキスト Speed テキスト テキスト.")


## 26.7 要点

| テキスト | 学習 テキスト | テキスト | 性能 |
|---|---|---|---|
| Full FT | 100% | テキスト | テキスト |
| LoRA | ~1% | テキスト | テキスト |
| QLoRA | ~1% + 4-bit base | テキスト テキスト | テキスト テキスト |
| Adapter | ~5% | テキスト | テキスト |
| Prefix Tuning | ~0.1% | テキスト テキスト | テキスト テキスト |

## 演習問題

1. LoRA rank $r = 1, 4, 16, 64$テキスト 学習テキスト 性能テキスト 比較テキスト.
2. LoRAテキスト QKV テキスト テキスト テキスト Qテキスト テキスト テキスト 比較テキスト.
3. 50%, 70%, 90% テキスト モデルテキスト テキスト度テキスト 比較テキスト.
4. Full FT vs LoRAテキスト 学習 時間テキスト テキスト テキスト.
5. LoRAテキスト テキスト テキスト テキスト テキスト テキスト テキスト.

> 解答: `solutions/ch26_solutions.ipynb`
